# Create Takeda Science Foundation Awards (武田科学振興財団)

Creates Takeda Science Foundation grants from the foundation's own grantee
database ("研究助成対象者データベース", https://file.takeda-sci.or.jp/bopdb/).
An empty search returns every grantee; each record has the grant number
(`detail.php?no=`), research title, programme, institution, department,
applicant (family/given already split by the database), fiscal year, and for
older grants keywords and a results summary. **8,449 grants, FY2009-2025, 100%
title/PI/institution/year** (local run 2026-09-30). Titles and programme names
are in Japanese as published.

Programmes include 武田報彰医学研究助成, 生命科学研究助成, ライフサイエンス研究助成,
医学系研究助成 (by field), 薬学系研究助成, 特定研究助成, ビジョナリーリサーチ助成, the
"継続" (continuation) grants, ハイリスク新興感染症研究助成, and the school science
education grants (小学校/中学校/高等学校理科教育振興助成, 1,171 grants to schools),
which ship with `funding_type = 'grant'` rather than `'research'`.

**Prerequisites:**
- Run `scripts/local/takeda_science_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/takeda_science_foundation/takeda_science_foundation_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the database's grant number (10 digits,
`{fiscal year}{programme}{serial}`, e.g. `2024013172`), the only identifier
the foundation publishes. Citation-side check
(`grants.funder:F4320309807` award ids in `openalex_awards_raw`): the ~1,700
acknowledgement shells carry other funders' ids (KAKENHI `JP22K06639`, AMED
`JP21dm0107071`, ...) or none; no Takeda-specific id form exists to match.

**Amounts:** not published per grant (fixed per programme in the call
documents), so `amount`/`currency` are NULL - §6.7 waived.
**Dates:** only the fiscal year (申請年度) is published; `start_year` = that year and
`start_date` = Jan 1 of it (runbook §2.3 year-only pattern). No end date.

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320309807`,
metadata read from `openalex.funders.funders` (ror 02y123g31, doi 10.13039/100007449).

**Priority:** `502` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.takeda_science_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/takeda_science_foundation/takeda_science_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants FROM openalex.awards.takeda_science_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.takeda_science_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.takeda_science_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320309807 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320309807;

## Step 2: Create Takeda Science Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.takeda_science_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320309807  -- Takeda Science Foundation
),

g AS (
    SELECT
        r.*,
        TRY_CAST(r.fiscal_year AS INT) AS fy,
        NULLIF(TRIM(r.lead_given_name), '') AS given,
        NULLIF(TRIM(r.lead_family_name), '') AS family
    FROM openalex.awards.takeda_science_foundation_raw r
    WHERE r.grant_no IS NOT NULL AND TRIM(r.grant_no) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.grant_no))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.result_summary), '') as description,
    f.funder_id,
    TRIM(g.grant_no) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    -- 11/12/19 = high-school / junior-high / elementary science-education grants
    CASE WHEN g.program_code IN ('11', '12', '19') THEN 'grant' ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'takeda_bopdb' as provenance,
    CASE WHEN g.fy IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.fy, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.fy as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', CONCAT_WS(' ', NULLIF(TRIM(g.institution), ''), NULLIF(TRIM(g.department), '')),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.grant_no))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is a 10-digit DB grant number starting with its fiscal year.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^(19|20)[0-9]{8}$' AND LEFT(funder_award_id, 4) = CAST(start_year AS STRING) THEN 0 ELSE 1 END) AS bad_shape
FROM openalex.awards.takeda_science_foundation_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'takeda_bopdb' AND priority = 502;

-- Insert into openalex_awards_raw with priority.
-- Priority 502: direct-from-funder ingest of the Takeda Science Foundation grantee DB.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    502 as priority
FROM openalex.awards.takeda_science_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.takeda_science_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.takeda_science_foundation_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.takeda_science_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail
-- (continuation grants mean a few PIs legitimately hold several).
SELECT lead_investigator.family_name AS family, lead_investigator.given_name AS given, COUNT(*) AS n
FROM openalex.awards.takeda_science_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (expect 100% title/PI/year; ~18% description; amount waived).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.takeda_science_foundation_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'takeda_bopdb'
GROUP BY provenance, priority;